<a href="https://colab.research.google.com/github/JTMouat/proyecto-ia-mouat/blob/main/Deteccion_Gestos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
#Revisa memoria disponible

!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   44G   64G  41% /


In [27]:
#Detecta si el dataset ya esta cargado para no tener que descargar denuevo

import os

if not os.path.exists('/content/hagrid'):
    # aquí van las celdas de descarga y descompresión
    ...
else:
    print("El dataset ya está en esta sesión")

El dataset ya está en esta sesión


In [ ]:
#Descarga el dataset de imagenes

URL = "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid/hagrid_dataset_new_554800/hagrid_dataset_512.zip"
!wget -c -O /content/hagrid512.zip "{URL}"

--2026-10-07 21:43:59--  https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid/hagrid_dataset_new_554800/hagrid_dataset_512.zip
Resolving rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru (rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru)... 46.243.206.34, 46.243.206.35
Connecting to rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru (rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru)|46.243.206.34|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 23474033757 (22G), 16294958173 (15G) remaining [application/zip]
Saving to: ‘/content/hagrid512.zip’

/content/hagrid512. 100%[++++++=============>]  21.86G  11.3MB/s    in 23m 34s 

2026-10-07 22:07:39 (11.0 MB/s) - ‘/content/hagrid512.zip’ saved [23474033757/23474033757]



In [ ]:
#Descomprime y borra el zip del dataset

!unzip -q /content/hagrid512.zip -d /content/hagrid
!rm /content/hagrid512.zip
!du -sh /content/hagrid
!ls /content/hagrid | head

24G	/content/hagrid
call
dislike
fist
four
like
mute
ok
one
palm
peace


In [28]:
#Descarga y descomprime el zip con las anotaciones del dataset

ANN_URL = "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/hagrid/hagrid_dataset_new_554800/annotations.zip"
!wget -q -O /content/annotations.zip "{ANN_URL}"
!unzip -q /content/annotations.zip -d /content/ann

replace /content/ann/train/four.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [29]:
#Comprobacion

print(" -----------")
print("|    Test   |")
print(" -----------")
!ls /content/ann/test
print()
print(" -----------")
print("|   Train   |")
print(" -----------")
!ls /content/ann/train
print()
print(" -----------")
print("|    Val    |")
print(" -----------")
!ls /content/ann/val

 -----------
|    Test   |
 -----------
call.json     mute.json		   peace.json	       three.json
dislike.json  ok.json		   rock.json	       two_up_inverted.json
fist.json     one.json		   stop_inverted.json  two_up.json
four.json     palm.json		   stop.json
like.json     peace_inverted.json  three2.json

 -----------
|   Train   |
 -----------
call.json     mute.json		   peace.json	       three.json
dislike.json  ok.json		   rock.json	       two_up_inverted.json
fist.json     one.json		   stop_inverted.json  two_up.json
four.json     palm.json		   stop.json
like.json     peace_inverted.json  three2.json

 -----------
|    Val    |
 -----------
call.json     mute.json		   peace.json	       three.json
dislike.json  ok.json		   rock.json	       two_up_inverted.json
fist.json     one.json		   stop_inverted.json  two_up.json
four.json     palm.json		   stop.json
like.json     peace_inverted.json  three2.json


In [32]:
import glob
import json
import os

# Buscar todos los archivos .json recursivamente dentro de /content/ann
ann_files = glob.glob("/content/ann/**/*.json", recursive=True)

print(f"Total de archivos JSON encontrados: {len(ann_files)}")
print("\nRutas de los archivos encontrados:")
for f in ann_files[:10]:  # Mostrar los primeros 10
    print(" -", f)

# Leer una muestra del primer JSON encontrado para verificar la estructura
if ann_files:
    sample_json = ann_files[0]
    print(f"\nInspeccionando contenido de: {sample_json}")
    with open(sample_json, "r") as f:
        data = json.load(f)

    first_key = list(data.keys())[0]
    print("\nEjemplo de entrada en el JSON:")
    print(json.dumps(data[first_key], indent=2))

Total de archivos JSON encontrados: 54

Rutas de los archivos encontrados:
 - /content/ann/val/stop.json
 - /content/ann/val/peace_inverted.json
 - /content/ann/val/two_up_inverted.json
 - /content/ann/val/rock.json
 - /content/ann/val/four.json
 - /content/ann/val/like.json
 - /content/ann/val/dislike.json
 - /content/ann/val/three.json
 - /content/ann/val/fist.json
 - /content/ann/val/mute.json

Inspeccionando contenido de: /content/ann/val/stop.json

Ejemplo de entrada en el JSON:
{
  "bboxes": [
    [
      0.23497881,
      0.84774288,
      0.06646489,
      0.11426561
    ],
    [
      0.57144595,
      0.33876474,
      0.10773131,
      0.14217515
    ]
  ],
  "user_id": "192b43d585030ff103c9bbdc918aee9d4d0cf442e8cb6e26bd9faf1be893a546",
  "labels": [
    "no_gesture",
    "stop"
  ]
}


In [33]:
import glob
import json
import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 1. Definir los gestos para el subset
SELECTED_GESTURES = ["stop", "fist", "like", "one", "palm", "peace"]
dataset_rows = []

# Obtener archivos JSON en /content/ann
all_json_files = glob.glob("/content/ann/**/*.json", recursive=True)

for json_path in all_json_files:
    with open(json_path, "r") as f:
        data = json.load(f)

    for img_id, info in data.items():
        user_id = info.get("user_id", "unknown")
        bboxes = info.get("bboxes", [])
        labels = info.get("labels", [])

        # Recorrer cada mano detectada en la imagen
        for bbox, label in zip(bboxes, labels):
            if label in SELECTED_GESTURES:

                img_path = os.path.join(
                    "/content/hagrid", label, f"{img_id}.jpg"
                )

                dataset_rows.append({
                    "image_id": img_id,
                    "gesture": label,
                    "user_id": user_id,
                    "bbox": bbox,  # Bounding box correspondiente al gesto exacto
                    "image_path": img_path,
                })

df = pd.DataFrame(dataset_rows)

# Eliminar posibles duplicados de JSON entre carpetas
df = df.drop_duplicates(
    subset=["image_id", "gesture", "user_id"]
).reset_index(drop=True)

# Verificar existencia real de los archivos de imagen en disco
df["exists"] = df["image_path"].apply(os.path.exists)
df_valid = df[df["exists"]].reset_index(drop=True)

print(f"Total de instancias de gestos encontradas: {len(df)}")
print(f"Instancias con imagen existente en disco: {len(df_valid)}")

# Mostrar desglose por gesto
print("\nDistribución por gesto:")
print(df_valid["gesture"].value_counts())

# 2. Split por Usuario (GroupShuffleSplit)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, temp_idx = next(gss.split(df_valid, groups=df_valid["user_id"]))

train_df = df_valid.iloc[train_idx].reset_index(drop=True)
temp_df = df_valid.iloc[temp_idx].reset_index(drop=True)

# Split del 30% restante en 15% Val y 15% Test
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df["user_id"]))

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print("\n--- Split por Usuario Completado ---")
print(
    f"Train: {len(train_df)} muestras | {train_df['user_id'].nunique()} usuarios"
)
print(
    f"Val:   {len(val_df)} muestras | {val_df['user_id'].nunique()} usuarios"
)
print(
    f"Test:  {len(test_df)} muestras | {test_df['user_id'].nunique()} usuarios"
)

Total de instancias de gestos encontradas: 188650
Instancias con imagen existente en disco: 188650

Distribución por gesto:
gesture
peace    31801
palm     31710
fist     31543
stop     31268
like     31244
one      31084
Name: count, dtype: int64

--- Split por Usuario Completado ---
Train: 132032 muestras | 14070 usuarios
Val:   30009 muestras | 3015 usuarios
Test:  26609 muestras | 3015 usuarios


In [34]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

# 1. Submuestreo para entrenamiento rápido en Colab
# Mantenemos la estructura por usuarios para respetar el split
train_sub = train_df.groupby("gesture", group_keys=False).apply(
    lambda x: x.sample(n=2000, random_state=42)
).reset_index(drop=True)

val_sub = val_df.groupby("gesture", group_keys=False).apply(
    lambda x: x.sample(n=400, random_state=42)
).reset_index(drop=True)

test_sub = test_df.groupby("gesture", group_keys=False).apply(
    lambda x: x.sample(n=400, random_state=42)
).reset_index(drop=True)

print(f"Muestras de entrenamiento reducidas: {len(train_sub)}")
print(f"Muestras de validación reducidas:   {len(val_sub)}")

# Mapeo de etiquetas a índices
label2id = {gesture: i for i, gesture in enumerate(SELECTED_GESTURES)}
id2label = {i: gesture for gesture, i in label2id.items()}

# 2. Definición del Dataset con recorte (Crop) de la mano
class GestureCropDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        w, h = image.size

        # Las bboxes normalizadas [x, y, w, h] entre 0 y 1
        bx, by, bw, bh = row["bbox"]
        xmin = int(max(0, bx * w))
        ymin = int(max(0, by * h))
        xmax = int(min(w, (bx + bw) * w))
        ymax = int(min(h, (by + bh) * h))

        # Para crop inválido, usar imagen completa
        if xmax <= xmin or ymax <= ymin:
            crop = image
        else:
            crop = image.crop((xmin, ymin, xmax, ymax))

        if self.transform:
            crop = self.transform(crop)

        label = label2id[row["gesture"]]
        return crop, torch.tensor(label, dtype=torch.long)

# Transforms para data augmentation y normalización
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = GestureCropDataset(train_sub, transform=train_transform)
val_dataset = GestureCropDataset(val_sub, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

Muestras de entrenamiento reducidas: 12000
Muestras de validación reducidas:   2400


/tmp/ipykernel_3759/684874007.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_sub = train_df.groupby("gesture", group_keys=False).apply(
/tmp/ipykernel_3759/684874007.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sub = val_df.groupby("gesture", group_keys=False).apply(
/tmp/ipykernel_3759/684874007.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This 

In [36]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class GestureEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        # ResNet18 preentrenada en ImageNet
        base_model = resnet18(weights=ResNet18_Weights.DEFAULT)
        # Extraemos red eliminando última capa totalmente conectada (fc)
        self.backbone = nn.Sequential(*list(base_model.children())[:-1])
        self.embed_dim = embed_dim

    def forward(self, x):
        features = self.backbone(x)
        features = torch.flatten(features, 1)  # Vector de embeddings de dim 512
        return features

class GestureClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(encoder.embed_dim, num_classes)
        )

    def forward(self, x):
        embeddings = self.encoder(x)
        logits = self.classifier(embeddings)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = GestureEncoder()
model = GestureClassifier(encoder, num_classes=len(SELECTED_GESTURES)).to(device)

print(f"Modelo cargado en dispositivo: {device}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 54.8MB/s]


Modelo cargado en dispositivo: cpu


In [38]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from tqdm.auto import tqdm

# Mapeo de etiquetas
label2id = {gesture: i for i, gesture in enumerate(SELECTED_GESTURES)}
id2label = {i: gesture for gesture, i in label2id.items()}

# Subset reducido adecuado para CPU (500 imágenes por gesto = 3,000 totales)
train_cpu = train_df.groupby("gesture", group_keys=False).apply(
    lambda x: x.sample(n=500, random_state=42)
).reset_index(drop=True)

val_cpu = val_df.groupby("gesture", group_keys=False).apply(
    lambda x: x.sample(n=100, random_state=42)
).reset_index(drop=True)

test_cpu = test_df.groupby("gesture", group_keys=False).apply(
    lambda x: x.sample(n=100, random_state=42)
).reset_index(drop=True)

transform_pipeline = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def preload_images_to_ram(df, desc="Cargando en RAM"):
    images_list = []
    labels_list = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            img = Image.open(row["image_path"]).convert("RGB")
            w, h = img.size

            bx, by, bw, bh = row["bbox"]
            xmin, ymin = int(max(0, bx * w)), int(max(0, by * h))
            xmax, ymax = int(min(w, (bx + bw) * w)), int(min(h, (by + bh) * h))

            if xmax > xmin and ymax > ymin:
                crop = img.crop((xmin, ymin, xmax, ymax))
            else:
                crop = img

            tensor_img = transform_pipeline(crop)
            images_list.append(tensor_img)
            labels_list.append(label2id[row["gesture"]])
        except Exception:
            continue

    return torch.stack(images_list), torch.tensor(labels_list, dtype=torch.long)

# Cargar en RAM
print("Precargando imágenes en la memoria RAM...")
X_train, y_train = preload_images_to_ram(train_cpu, desc="Entrenamiento")
X_val, y_val = preload_images_to_ram(val_cpu, desc="Validación")

# DataLoaders directamente desde memoria RAM
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("\n¡Carga completada con éxito en la memoria RAM!")

Precargando imágenes en la memoria RAM...


/tmp/ipykernel_3759/837085717.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_cpu = train_df.groupby("gesture", group_keys=False).apply(
/tmp/ipykernel_3759/837085717.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_cpu = val_df.groupby("gesture", group_keys=False).apply(
/tmp/ipykernel_3759/837085717.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This

Entrenamiento:   0%|          | 0/3000 [00:00<?, ?it/s]

Validación:   0%|          | 0/600 [00:00<?, ?it/s]


¡Carga completada con éxito en la memoria RAM!


In [39]:
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights

# 1. Definir Encoder y Clasificador
class GestureEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        base_model = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base_model.children())[:-1])
        self.embed_dim = embed_dim

    def forward(self, x):
        features = self.backbone(x)
        return torch.flatten(features, 1)

class GestureClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(encoder.embed_dim, num_classes)
        )

    def forward(self, x):
        embeddings = self.encoder(x)
        return self.classifier(embeddings)

device = torch.device("cpu")
encoder = GestureEncoder()
model = GestureClassifier(encoder, num_classes=len(SELECTED_GESTURES)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# 2. Entrenamiento con barra de progreso
EPOCHS = 3  # 3 épocas: suficientes para afinar capa superior y extraer un buen encoder

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Época {epoch+1}/{EPOCHS} [CPU]", leave=True)
    for images, labels in pbar:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    epoch_acc = (correct / total) * 100

    # Validación
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = (val_correct / val_total) * 100
    print(f"➡️ Época {epoch+1} Finalizada | Train Acc: {epoch_acc:.2f}% | Val Acc: {val_acc:.2f}%\n")

# 3. Guardar encoder entrenado
torch.save(model.encoder.state_dict(), "gesture_encoder_hito1.pth")
print("¡Hito 1 completado exitosamente! Pesos guardados en 'gesture_encoder_hito1.pth'.")

Época 1/3 [CPU]:   0%|          | 0/94 [00:00<?, ?it/s]

➡️ Época 1 Finalizada | Train Acc: 87.00% | Val Acc: 98.33%



Época 2/3 [CPU]:   0%|          | 0/94 [00:00<?, ?it/s]

➡️ Época 2 Finalizada | Train Acc: 99.30% | Val Acc: 99.33%



Época 3/3 [CPU]:   0%|          | 0/94 [00:00<?, ?it/s]

➡️ Época 3 Finalizada | Train Acc: 99.73% | Val Acc: 99.50%

¡Hito 1 completado exitosamente! Pesos guardados en 'gesture_encoder_hito1.pth'.


In [41]:
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset

# 1. Precargar el conjunto de Test en RAM
X_test, y_test = preload_images_to_ram(test_cpu, desc="Test")
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. Evaluar el modelo
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
  for images, labels in test_loader:
    outputs = model(images)
    _, preds = torch.max(outputs, 1)
    all_preds.extend(preds.numpy())
    all_labels.extend(labels.numpy())

# 3. Mostrar el reporte final de métricas
target_names = [id2label[i] for i in range(len(SELECTED_GESTURES))]
print("==================================================")
print("   REPORTE DE CLASIFICACIÓN - CONJUNTO DE TEST   ")
print("==================================================")
print(classification_report(all_labels, all_preds, target_names=target_names))

Test:   0%|          | 0/600 [00:00<?, ?it/s]

   REPORTE DE CLASIFICACIÓN - CONJUNTO DE TEST   
              precision    recall  f1-score   support

        stop       0.96      0.98      0.97       100
        fist       1.00      1.00      1.00       100
        like       0.98      0.97      0.97       100
         one       0.96      0.94      0.95       100
        palm       0.97      0.95      0.96       100
       peace       0.94      0.97      0.96       100

    accuracy                           0.97       600
   macro avg       0.97      0.97      0.97       600
weighted avg       0.97      0.97      0.97       600



In [42]:
import os

# Guardar los DataFrames de los subsets a CSV
train_cpu.to_csv("/content/train_subset.csv", index=False)
val_cpu.to_csv("/content/val_subset.csv", index=False)
test_cpu.to_csv("/content/test_subset.csv", index=False)

print("--- Archivos finales generados para el Hito 1 ---")
files_to_check = [
    "gesture_encoder_hito1.pth",
    "train_subset.csv",
    "val_subset.csv",
    "test_subset.csv",
]

for f in files_to_check:
  if os.path.exists(f):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"✅ {f} ({size_mb:.2f} MB)")
  else:
    print(f"❌ Falta: {f}")

--- Archivos finales generados para el Hito 1 ---
✅ gesture_encoder_hito1.pth (42.71 MB)
✅ train_subset.csv (0.64 MB)
✅ val_subset.csv (0.13 MB)
✅ test_subset.csv (0.13 MB)


In [55]:
from PIL import Image
import torch
#Prueba con fotos propias

foto_path = "/content/fotos prueba/ft1.jpg"
imagen = Image.open(foto_path).convert("RGB")

tensor_imagen = val_transform(imagen).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
  outputs = model(tensor_imagen)
  _, pred_idx = torch.max(outputs, 1)

gesto_detectado_ft1 = id2label[pred_idx.item()]

#------------------------------------------------------------------------------------------

foto_path = "/content/fotos prueba/ft2.jpg"
imagen = Image.open(foto_path).convert("RGB")

tensor_imagen = val_transform(imagen).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
  outputs = model(tensor_imagen)
  _, pred_idx = torch.max(outputs, 1)

gesto_detectado_ft2 = id2label[pred_idx.item()]

#------------------------------------------------------------------------------------------

foto_path = "/content/fotos prueba/ft3.jpg"
imagen = Image.open(foto_path).convert("RGB")

tensor_imagen = val_transform(imagen).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
  outputs = model(tensor_imagen)
  _, pred_idx = torch.max(outputs, 1)

gesto_detectado_ft3 = id2label[pred_idx.item()]

#------------------------------------------------------------------------------------------

foto_path = "/content/fotos prueba/ft4.jpg"
imagen = Image.open(foto_path).convert("RGB")

tensor_imagen = val_transform(imagen).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
  outputs = model(tensor_imagen)
  _, pred_idx = torch.max(outputs, 1)

gesto_detectado_ft4 = id2label[pred_idx.item()]

#------------------------------------------------------------------------------------------

foto_path = "/content/fotos prueba/ft5.jpg"
imagen = Image.open(foto_path).convert("RGB")

tensor_imagen = val_transform(imagen).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
  outputs = model(tensor_imagen)
  _, pred_idx = torch.max(outputs, 1)

gesto_detectado_ft5 = id2label[pred_idx.item()]

#------------------------------------------------------------------------------------------

foto_path = "/content/fotos prueba/ft6.jpg"
imagen = Image.open(foto_path).convert("RGB")

tensor_imagen = val_transform(imagen).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
  outputs = model(tensor_imagen)
  _, pred_idx = torch.max(outputs, 1)

gesto_detectado_ft6 = id2label[pred_idx.item()]

#------------------------------------------------------------------------------------------
print(" ====================================")
print("|  Deteccion Gestos en fotos propias |")
print(" ====================================")
print()
print("Foto 1:")
print()
print("Gesto real: Fist")
print(f"Gesto identificado: {gesto_detectado_ft1.upper()} ✅")
print()
print("Foto 2:")
print()
print("Gesto real: Like")
print(f"Gesto identificado: {gesto_detectado_ft2.upper()} ✅")
print()
print("Foto 3:")
print()
print("Gesto real: One")
print(f"Gesto identificado: {gesto_detectado_ft3.upper()} ✅")
print()
print("Foto 4:")
print()
print("Gesto real: Palm")
print(f"Gesto identificado: {gesto_detectado_ft4.upper()} \U0000274C")
print()
print("Foto 5:")
print()
print("Gesto real: Peace")
print(f"Gesto identificado: {gesto_detectado_ft5.upper()} ✅")
print()
print("Foto 6:")
print()
print("Gesto real: Stop")
print(f"Gesto identificado: {gesto_detectado_ft6.upper()} \U0000274C")




|  Deteccion Gestos en fotos propias |

Foto 1:

Gesto real: Fist
Gesto identificado: FIST ✅

Foto 2:

Gesto real: Like
Gesto identificado: LIKE ✅

Foto 3:

Gesto real: One
Gesto identificado: ONE ✅

Foto 4:

Gesto real: Palm
Gesto identificado: LIKE ❌

Foto 5:

Gesto real: Peace
Gesto identificado: PEACE ✅

Foto 6:

Gesto real: Stop
Gesto identificado: PALM ❌
